In [ ]:
"""
Mortality Prediction with Chronos-2 Embeddings
Modularized and Validated Implementation
"""

import scipy.io
import torch
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
import pickle as pl
import glob
import traceback
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


In [ ]:
# ekpping a buffer for about 1 hour for the gap between vt and birth and vt and age at death

In [ ]:
# Load pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", 
                                            device_map="cpu", 
                                            dtype=torch.bfloat16)

# Model and data constants
CONTEXT_SIZE_HOURS = 80
CONTEXT_SIZE_SECONDS = CONTEXT_SIZE_HOURS * 3600  # 288,000 seconds
WINDOW_SIZE_SECONDS = 300  # 5 minutes
NUM_WINDOWS = CONTEXT_SIZE_SECONDS // WINDOW_SIZE_SECONDS  # 960 windows

# Thresholds for prediction
THRESHOLDS_DAYS = [3, 5, 7]
THRESHOLDS_SECONDS = [d * 86400 for d in THRESHOLDS_DAYS]

# Number of windows to generate per baby
NUM_LABEL_0_NICU = 10        # Label 0 windows for NICU babies (before threshold)
NUM_LABEL_1_NICU = 10        # Label 1 windows for NICU babies (within threshold)
NUM_LABEL_0_NON_NICU = 10    # Label 0 windows for non-NICU babies (survived)

In [ ]:
# ============================================================================
# STAGE 1: RAW DATA EXTRACTION
# ============================================================================

def extract_raw_timeseries(mat_file_path, start_time, context_size):
    """
    Extract raw HR and SPO2 time series data from MATLAB file
    
    Args:
        mat_file_path: Path to .mat file
        start_time: Start timestamp in seconds
        context_size: Duration of context window in seconds
    
    Returns:
        dict with keys: 'timestamps', 'hr_raw', 'spo2_raw', 'birth_time', 'death_ts'
        Returns None if insufficient data
    """
    try:
        mat_data = scipy.io.loadmat(mat_file_path)
    except Exception as e:
        print(f"Error loading {mat_file_path}: {e}")
        return None
    
    # Extract metadata
    birth_time = mat_data['pdata'][0][1]
    age_at_death = mat_data['pdata'][0][-1]
    death_ts = age_at_death * 86400 if not np.isnan(age_at_death) else None
    
    # Extract time series
    hr_cols = mat_data['vdata'][:,0]
    spo2_cols = mat_data['vdata'][:,2]
    timestamps = mat_data['vt'].flatten()
    
    # Filter by time range
    end_time = start_time + context_size
    time_mask = (timestamps >= start_time) & (timestamps < end_time)
    
    # Filter out NaN values
    valid_mask = ~np.isnan(hr_cols) & ~np.isnan(spo2_cols)
    final_mask = time_mask & valid_mask
    
    timestamps_valid = timestamps[final_mask]
    hr_valid = hr_cols[final_mask]
    spo2_valid = spo2_cols[final_mask]
    
    if len(timestamps_valid) == 0:
        return None
    
    return {
        'timestamps': timestamps_valid,
        'hr_raw': hr_valid,
        'spo2_raw': spo2_valid,
        'birth_time': birth_time,
        'death_ts': death_ts,
        'start_time': start_time,
        'context_size': context_size
    }


In [ ]:
# ============================================================================
# STAGE 2: WINDOWING (AGGREGATION)
# ============================================================================

def create_windowed_features_from_raw(raw_data, window_size):
    """
    Convert raw time series into fixed-size windowed features
    
    Args:
        raw_data: dict from extract_raw_timeseries()
        window_size: size of each window in seconds (default: 300s = 5 min)
    
    Returns:
        tuple: (hr_windows, spo2_windows) - numpy arrays of shape (num_windows,)
        Each window contains mean value of raw data in that time bin
    """
    timestamps = raw_data['timestamps']
    hr_raw = raw_data['hr_raw']
    spo2_raw = raw_data['spo2_raw']
    start_time = raw_data['start_time']
    context_size = raw_data['context_size']
    
    # Assign each timestamp to a window bin
    window_indices = np.floor((timestamps - start_time) / window_size).astype(int)
    num_windows = int(np.ceil(context_size / window_size))
    
    # Initialize with NaN
    hr_windows = np.full(num_windows, np.nan)
    spo2_windows = np.full(num_windows, np.nan)
    
    # Aggregate by mean within each window
    for idx in range(num_windows):
        mask = window_indices == idx
        if np.any(mask):
            hr_windows[idx] = np.mean(hr_raw[mask])
            spo2_windows[idx] = np.mean(spo2_raw[mask])
    
    return hr_windows, spo2_windows

def create_windowed_features(mat_file_path, start_time, context_size, window_size):
    """
    Convenience function: combines raw extraction + windowing
    
    Returns:
        tuple: (hr_windows, spo2_windows) or None if insufficient data
    """
    raw_data = extract_raw_timeseries(mat_file_path, start_time, context_size)
    if raw_data is None:
        return None
    return create_windowed_features_from_raw(raw_data, window_size)

In [ ]:
# ============================================================================
# STAGE 3: CHRONOS INPUT PREPARATION
# ============================================================================

def prepare_chronos_input(hr_windows, spo2_windows):
    """
    Prepare windowed data for Chronos model input
    
    Args:
        hr_windows: numpy array of HR values per window
        spo2_windows: numpy array of SPO2 values per window
    
    Returns:
        torch.Tensor of shape (2, 1, num_windows) ready for Chronos
        NaN values are imputed with mean
    """
    # Handle NaN values with mean imputation
    hr_clean = np.where(
        np.isnan(hr_windows), 
        np.nanmean(hr_windows) if not np.all(np.isnan(hr_windows)) else 0, 
        hr_windows
    ).astype(np.float32)
    
    spo2_clean = np.where(
        np.isnan(spo2_windows), 
        np.nanmean(spo2_windows) if not np.all(np.isnan(spo2_windows)) else 0, 
        spo2_windows
    ).astype(np.float32)
    
    # Create tensors
    hr_tensor = torch.tensor(hr_clean, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    spo2_tensor = torch.tensor(spo2_clean, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    
    # Concatenate: shape (2, 1, num_windows)
    context = torch.cat([hr_tensor, spo2_tensor], dim=0)
    
    return context


In [ ]:
def get_chronos_embedding(chronos_input, pipeline):
    """
    Generate embedding from Chronos model
    
    Args:
        chronos_input: torch.Tensor of shape (2, 1, num_windows)
        pipeline: Chronos2Pipeline model
    
    Returns:
        numpy array of shape (1536,) - concatenated HR + SPO2 embeddings
    """
    with torch.no_grad():
        embeddings, _ = pipeline.embed(chronos_input)
    
    # Extract last token embedding for each channel
    hr_emb = embeddings[0][0][-1].float()    # Shape: (768,)
    spo2_emb = embeddings[1][0][-1].float()  # Shape: (768,)
    
    # Concatenate both embeddings
    combined_embedding = torch.cat([hr_emb, spo2_emb], dim=0).cpu().numpy()
    
    return combined_embedding

def get_embedding(hr_windows, spo2_windows, pipeline):
    """
    Convenience function: combines preparation + embedding generation
    
    Returns:
        numpy array of embedding
    """
    chronos_input = prepare_chronos_input(hr_windows, spo2_windows)
    return get_chronos_embedding(chronos_input, pipeline)

In [ ]:
def generate_non_nicu_embeddings(mat_file_path, pipeline, num_windows=NUM_LABEL_0_NON_NICU):
    """
    Generate label 0 embeddings for non-NICU babies (survived)
    
    Strategy:
        - Non-overlapping windows from birth to end of recording
        - All windows labeled 0 (baby survived)
    
    Args:
        mat_file_path: Path to .mat file
        pipeline: Chronos2Pipeline model
        num_windows: Maximum number of windows to generate
    
    Returns:
        list of tuples: [(embedding, label), ...]
    """
    mat_data = scipy.io.loadmat(mat_file_path)
    birth_time = mat_data['pdata'][0][1]
    
    if np.isnan(birth_time):
        return []
    
    timestamps = mat_data['vt'].flatten()
    last_ts = timestamps[-1].item()
    
    datapoints = []
    
    # Generate non-overlapping windows from birth to end of recording
    available = last_ts - birth_time
    if available >= CONTEXT_SIZE_SECONDS:
        max_windows = min(num_windows, int(available / CONTEXT_SIZE_SECONDS))
        step = available / max_windows  # Non-overlapping step
        
        for i in range(max_windows):
            start = birth_time + i * step
            result = create_windowed_features(mat_file_path, start, 
                                             CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
            if result is not None:
                embedding = get_embedding(result[0], result[1], pipeline)
                datapoints.append((embedding, 0))  # Label 0: survived
    
    return datapoints

In [ ]:
def generate_nicu_embeddings(mat_file_path, pipeline, threshold_seconds, 
                             num_label_0=NUM_LABEL_0_NICU, 
                             num_label_1=NUM_LABEL_1_NICU):
    """
    Generate embeddings for NICU babies (died) - threshold-dependent
    
    Strategy:
        - Label 0: Overlapping windows BEFORE threshold boundary
                  (baby survives > threshold days after window)
        - Label 1: Overlapping windows WITHIN threshold period
                  (baby dies ≤ threshold days after window)
    
    Args:
        mat_file_path: Path to .mat file
        pipeline: Chronos2Pipeline model
        threshold_seconds: Prediction threshold in seconds
        num_label_0: Number of label 0 windows to generate
        num_label_1: Number of label 1 windows to generate
    
    Returns:
        list of tuples: [(embedding, label), ...]
    """
    mat_data = scipy.io.loadmat(mat_file_path)
    age_at_death = mat_data['pdata'][0][-1]
    birth_time = mat_data['pdata'][0][1]
    
    if np.isnan(age_at_death) or np.isnan(birth_time):
        return []
    
    death_ts = age_at_death * 86400
    threshold_start = death_ts - threshold_seconds  # Boundary between label 0 and label 1
    
    datapoints = []
    
    # ========================================================================
    # Label 0: OVERLAPPING windows BEFORE threshold
    # Condition: window_end + threshold_seconds < death_ts
    # ========================================================================
    available = threshold_start - birth_time
    if available >= CONTEXT_SIZE_SECONDS and num_label_0 > 0:
        # Create overlapping windows
        step = (available - CONTEXT_SIZE_SECONDS) / (num_label_0 - 1) if num_label_0 > 1 else 0
        
        for i in range(num_label_0):
            start = birth_time + i * step
            result = create_windowed_features(mat_file_path, start, 
                                             CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
            if result is not None:
                embedding = get_embedding(result[0], result[1], pipeline)
                datapoints.append((embedding, 0))  # Label 0: safe period
    
    # ========================================================================
    # Label 1: OVERLAPPING windows WITHIN threshold period
    # Condition: window_end + threshold_seconds ≥ death_ts
    # ========================================================================
    if num_label_1 > 0:
        # Create overlapping windows ending at death
        # Step backward from death to create overlapping windows
        step = (threshold_seconds - CONTEXT_SIZE_SECONDS) / (num_label_1 - 1) if num_label_1 > 1 else 0
        step = max(step, 0)  # Ensure non-negative (handles threshold < context)
        
        for i in range(num_label_1):
            # Start from (death - context_size) and step backward
            start = death_ts - CONTEXT_SIZE_SECONDS - i * step
            
            # Ensure window is within valid range
            start = max(birth_time, start)  # Don't go before birth
            
            # Only add if window overlaps with threshold period
            if start + CONTEXT_SIZE_SECONDS > threshold_start:
                result = create_windowed_features(mat_file_path, start, 
                                                 CONTEXT_SIZE_SECONDS, WINDOW_SIZE_SECONDS)
                if result is not None:
                    embedding = get_embedding(result[0], result[1], pipeline)
                    datapoints.append((embedding, 1))  # Label 1: danger period
    
    return datapoints

In [ ]:
def run_experiment(nicu_files, non_nicu_embeddings_label0, pipeline, 
                  threshold_days, threshold_seconds):
    """
    Run experiment for one threshold
    
    Args:
        nicu_files: List of NICU .mat file paths
        non_nicu_embeddings_label0: Pre-computed non-NICU embeddings (label 0)
        pipeline: Chronos2Pipeline model
        threshold_days: Threshold in days (for logging)
        threshold_seconds: Threshold in seconds (for computation)
    
    Returns:
        tuple: (embeddings, labels) - lists of embeddings and corresponding labels
    """
    print(f"\n{'='*60}")
    print(f"Threshold: <{threshold_days} days")
    print(f"{'='*60}")
    
    embeddings, labels = [], []
    error_count = 0
    
    # Add pre-computed non-NICU embeddings (label 0)
    for emb, lbl in non_nicu_embeddings_label0:
        embeddings.append(emb)
        labels.append(lbl)
    
    print(f"Added {len(non_nicu_embeddings_label0)} non-NICU label 0 embeddings")
    
    # Process NICU files (threshold-dependent)
    print(f"\nProcessing {len(nicu_files)} NICU files...")
    for i, file_path in enumerate(nicu_files):
        baby_id = file_path.split('/')[-1].split('_')[1]
        print(f"{i+1}/{len(nicu_files)}: Baby {baby_id}...", end=" ")
        
        try:
            datapoints = generate_nicu_embeddings(file_path, pipeline, threshold_seconds)
            for emb, lbl in datapoints:
                embeddings.append(emb)
                labels.append(lbl)
            
            label_0_count = sum(1 for _, l in datapoints if l == 0)
            label_1_count = sum(1 for _, l in datapoints if l == 1)
            print(f"{len(datapoints)} points (label 0: {label_0_count}, label 1: {label_1_count})")
        except Exception as e:
            error_count += 1
            print(f"\nERROR: {str(e)}")
            traceback.print_exc()
    
    # Summary
    print(f"\n{'='*60}")
    print(f"Total: {len(embeddings)} datapoints")
    print(f"Label 0: {sum(l==0 for l in labels)} | Label 1: {sum(l==1 for l in labels)}")
    print(f"Errors: {error_count}")
    print(f"{'='*60}")
    
    # Save embeddings
    with open(f'embeddings_{threshold_days}d.pkl', 'wb') as f:
        pl.dump({'embeddings': embeddings, 'labels': labels}, f)
    
    return embeddings, labels

In [ ]:
class EmbeddingDataset(Dataset):
    """PyTorch Dataset for embeddings"""
    def __init__(self, embeddings, labels):
        self.embeddings = [torch.FloatTensor(e) for e in embeddings]
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

class Classifier(nn.Module):
    """
    Neural network classifier for embeddings
    
    Architecture:
        Input (1536) → Hidden (128) → Hidden (64) → Output (2)
        With ReLU activations and 30% dropout
    """
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, 2)
        )
    
    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    for embs, labels in loader:
        embs, labels = embs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(embs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, device):
    """Validate and compute metrics"""
    model.eval()
    preds, lbls, probs = [], [], []
    with torch.no_grad():
        for embs, labels in loader:
            embs, labels = embs.to(device), labels.to(device)
            outputs = model(embs)
            probs.extend(torch.softmax(outputs, 1)[:, 1].cpu().numpy())
            preds.extend(outputs.argmax(1).cpu().numpy())
            lbls.extend(labels.cpu().numpy())
    
    return {
        'accuracy': accuracy_score(lbls, preds),
        'precision': precision_score(lbls, preds, zero_division=0),
        'recall': recall_score(lbls, preds, zero_division=0),
        'f1': f1_score(lbls, preds, zero_division=0),
        'auc': roc_auc_score(lbls, probs) if len(set(lbls)) > 1 else 0
    }

def k_fold_cv(embeddings, labels, k=5, epochs=50, batch_size=32, lr=0.001, device='cpu'):
    """
    K-fold cross validation
    
    Args:
        embeddings: List of embedding arrays
        labels: List of labels
        k: Number of folds
        epochs: Training epochs per fold
        batch_size: Batch size for training
        lr: Learning rate
        device: 'cpu' or 'cuda'
    
    Returns:
        tuple: (avg_metrics, std_metrics, fold_metrics)
    """
    if len(embeddings) == 0:
        print("No embeddings to train on!")
        return None, None, None
    
    embeddings = np.array(embeddings)
    labels = np.array(labels)
    dim = embeddings[0].flatten().shape[0]
    
    print(f"\nK-Fold CV: {len(labels)} samples, {k} folds, embedding dim={dim}")
    print(f"Class distribution: {np.bincount(labels)}")
    
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(embeddings, labels)):
        print(f"Fold {fold+1}/{k}...", end=" ")
        
        train_data = EmbeddingDataset(embeddings[train_idx], labels[train_idx])
        val_data = EmbeddingDataset(embeddings[val_idx], labels[val_idx])
        train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_data, batch_size=batch_size)
        
        model = Classifier(dim).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        for _ in range(epochs):
            train_epoch(model, train_loader, criterion, optimizer, device)
        
        metrics = validate(model, val_loader, device)
        fold_metrics.append(metrics)
        print(f"Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")
    
    # Average metrics across folds
    avg = {k: np.mean([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
    std = {k: np.std([f[k] for f in fold_metrics]) for k in fold_metrics[0]}
    
    print(f"\n{'='*60}")
    for k in avg:
        print(f"{k.capitalize():12s}: {avg[k]:.4f} ± {std[k]:.4f}")
    
    return avg, std, fold_metrics

In [ ]:

def stage1_generate_embeddings():
    """
    STAGE 1: Generate embeddings for all thresholds
    
    Outputs:
        - embeddings_3d.pkl  (contains ALL embeddings for 3-day threshold)
        - embeddings_5d.pkl  (contains ALL embeddings for 5-day threshold)
        - embeddings_7d.pkl  (contains ALL embeddings for 7-day threshold)
    """
    print("\n" + "="*80)
    print(" "*25 + "STAGE 1: EMBEDDING GENERATION")
    print("="*80 + "\n")
    
    # Load Chronos pipeline
    print("Loading Chronos-2 pipeline...")
    pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", 
                                                device_map="cpu", 
                                                dtype=torch.bfloat16)
    print("✓ Pipeline loaded\n")
    
    # Load valid baby IDs
    print("Loading valid baby IDs from valid_summary_dic.pkl...")
    with open('valid_summary_dic.pkl', 'rb') as f:
        valid_summary_dic = pl.load(f)
    
    nicu_mask = valid_summary_dic['indicator_vector'] == 1
    nicu_baby_ids = set(valid_summary_dic['baby_id'][nicu_mask])
    non_nicu_baby_ids = set(valid_summary_dic['baby_id'][~nicu_mask])
    
    print(f"✓ NICU babies (died): {len(nicu_baby_ids)}")
    print(f"✓ Non-NICU babies (survived): {len(non_nicu_baby_ids)}\n")
    
    # Filter mat files
    all_mat_files = sorted(glob.glob('./download/Vitals/*.mat'))
    nicu_files = [f for f in all_mat_files if f.split('/')[-1] in nicu_baby_ids]
    non_nicu_files = [f for f in all_mat_files if f.split('/')[-1] in non_nicu_baby_ids]
    
    print(f"✓ NICU mat files: {len(nicu_files)}")
    print(f"✓ Non-NICU mat files: {len(non_nicu_files)}\n")
    
    # ========================================================================
    # OPTIMIZATION: Generate non-NICU embeddings once (in memory, not saved)
    # ========================================================================
    print("─" * 80)
    print("Generating non-NICU embeddings (will be reused across thresholds)...")
    print("─" * 80)
    
    non_nicu_embeddings_label0 = []  # Temporary, in-memory only
    error_count = 0
    
    for i, file_path in enumerate(non_nicu_files):
        baby_id = file_path.split('/')[-1].split('_')[1]
        print(f"[{i+1}/{len(non_nicu_files)}] Baby {baby_id}...", end=" ")
        
        try:
            datapoints = generate_non_nicu_embeddings(file_path, pipeline)
            non_nicu_embeddings_label0.extend(datapoints)
            print(f"✓ {len(datapoints)} windows")
        except Exception as e:
            error_count += 1
            print(f"✗ ERROR: {str(e)}")
    
    print(f"\n✓ Generated {len(non_nicu_embeddings_label0)} non-NICU embeddings (in memory)")
    print(f"  Errors: {error_count}\n")
    
    # ========================================================================
    # Generate embeddings for each threshold (includes non-NICU + NICU)
    # ========================================================================
    for threshold_days, threshold_seconds in zip(THRESHOLDS_DAYS, THRESHOLDS_SECONDS):
        print("─" * 80)
        print(f"Generating embeddings for {threshold_days}-day threshold")
        print("─" * 80)
        
        embeddings, labels = [], []
        
        # Add non-NICU embeddings (label 0)
        for emb, lbl in non_nicu_embeddings_label0:
            embeddings.append(emb)
            labels.append(lbl)
        
        print(f"✓ Added {len(non_nicu_embeddings_label0)} non-NICU embeddings")
        
        # Process NICU files (threshold-dependent)
        print(f"  Processing {len(nicu_files)} NICU files...\n")
        error_count = 0
        
        for i, file_path in enumerate(nicu_files):
            baby_id = file_path.split('/')[-1].split('_')[1]
            print(f"  [{i+1}/{len(nicu_files)}] Baby {baby_id}...", end=" ")
            
            try:
                datapoints = generate_nicu_embeddings(file_path, pipeline, threshold_seconds)
                for emb, lbl in datapoints:
                    embeddings.append(emb)
                    labels.append(lbl)
                
                label_0_count = sum(1 for _, l in datapoints if l == 0)
                label_1_count = sum(1 for _, l in datapoints if l == 1)
                print(f"✓ {len(datapoints)} windows (L0:{label_0_count}, L1:{label_1_count})")
            except Exception as e:
                error_count += 1
                print(f"✗ ERROR: {str(e)}")
        
        # Summary
        print(f"\n  Summary for {threshold_days}-day threshold:")
        print(f"    Total embeddings: {len(embeddings)}")
        print(f"    Label 0: {sum(l==0 for l in labels)}")
        print(f"    Label 1: {sum(l==1 for l in labels)}")
        print(f"    Errors: {error_count}")
        
        # Save COMPLETE embeddings for this threshold
        output_file = f'embeddings_{threshold_days}d.pkl'
        with open(output_file, 'wb') as f:
            pl.dump({'embeddings': embeddings, 'labels': labels}, f)
        print(f"  ✓ Saved: {output_file}")
        print(f"    (Contains {len(non_nicu_embeddings_label0)} non-NICU + "
              f"{len(embeddings) - len(non_nicu_embeddings_label0)} NICU embeddings)\n")
    
    print("="*80)
    print(" "*15 + "✓ STAGE 1 COMPLETE: All embeddings generated")
    print(" "*20 + "(No separate non_nicu file needed)")
    print("="*80 + "\n\n")

In [ ]:
# precision and recall are calculated for the positive class (label 1) by default.


In [ ]:
import pickle as pl

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_3d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
arr

In [ ]:
arr

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_5d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
arr

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_7d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
arr

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/non_nicu_embeddings_label0.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
len(arr)

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/embeddings_3d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
sum(arr['labels'])

In [ ]:
len(arr["embeddings"])

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/embeddings_5d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
sum(arr['labels'])

In [ ]:
len(arr["embeddings"])

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/embeddings_7d.pkl','rb') as f:
    arr = pl.load(f)

In [ ]:
sum(arr['labels'])

In [ ]:
len(arr["embeddings"])

In [ ]:
# Load valid baby IDs
print("Loading valid baby IDs from valid_summary_dic.pkl...")
with open('valid_summary_dic.pkl', 'rb') as f:
    valid_summary_dic = pl.load(f)

# Split into NICU and non-NICU
nicu_mask = valid_summary_dic['indicator_vector'] == 1
nicu_baby_ids = set(valid_summary_dic['baby_id'][nicu_mask])
non_nicu_baby_ids = set(valid_summary_dic['baby_id'][~nicu_mask])

print(f"NICU babies (died): {len(nicu_baby_ids)}")
print(f"Non-NICU babies (survived): {len(non_nicu_baby_ids)}")

# Filter mat files
all_mat_files = sorted(glob.glob('./download/Vitals/*.mat'))
nicu_files = [f for f in all_mat_files if f.split('/')[-1] in nicu_baby_ids]
non_nicu_files = [f for f in all_mat_files if f.split('/')[-1] in non_nicu_baby_ids]


In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_3d.pkl','rb') as f:
    results = pl.load(f)

In [ ]:
results['avg']

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_5d.pkl','rb') as f:
    results = pl.load(f)

In [ ]:
results['avg']

In [ ]:
with open('/Users/adityagoyal/Desktop/Research - yin li/timeseries_with_chronos/results_7d.pkl','rb') as f:
    results = pl.load(f)

In [ ]:
results['avg']

In [ ]:
import pandas as pd
import pickle as pl

# Load all three threshold results
thresholds = [3, 5, 7]
all_results = {}

for days in thresholds:
    with open(f'results_{days}d.pkl', 'rb') as f:
        all_results[days] = pl.load(f)

# Create comparison table
rows = []
for days in thresholds:
    if all_results[days]['avg'] is not None:
        row = {
            'Threshold': f'<{days} days',
            'Accuracy': f"{all_results[days]['avg']['accuracy']:.4f} ± {all_results[days]['std']['accuracy']:.4f}",
            'Precision': f"{all_results[days]['avg']['precision']:.4f} ± {all_results[days]['std']['precision']:.4f}",
            'Recall': f"{all_results[days]['avg']['recall']:.4f} ± {all_results[days]['std']['recall']:.4f}",
            'F1-Score': f"{all_results[days]['avg']['f1']:.4f} ± {all_results[days]['std']['f1']:.4f}",
            'AUC': f"{all_results[days]['avg']['auc']:.4f} ± {all_results[days]['std']['auc']:.4f}"
        }
        rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)
print("\n" + "="*100)
print("MORTALITY PREDICTION RESULTS - Comparison Across Thresholds")
print("="*100)
print(df.to_string(index=False))
print("="*100)

# Optional: Save to CSV
df.to_csv('mortality_prediction_results.csv', index=False)
print("\nResults saved to 'mortality_prediction_results.csv'")

# Also create a table with just mean values (cleaner)
rows_mean = []
for days in thresholds:
    if all_results[days]['avg'] is not None:
        row = {
            'Threshold': f'<{days} days',
            'Accuracy': f"{all_results[days]['avg']['accuracy']:.4f}",
            'Precision': f"{all_results[days]['avg']['precision']:.4f}",
            'Recall': f"{all_results[days]['avg']['recall']:.4f}",
            'F1-Score': f"{all_results[days]['avg']['f1']:.4f}",
            'AUC': f"{all_results[days]['avg']['auc']:.4f}"
        }
        rows_mean.append(row)

df_mean = pd.DataFrame(rows_mean)
print("\n" + "="*80)
print("MEAN VALUES ONLY")
print("="*80)
print(df_mean.to_string(index=False))
print("="*80)

In [ ]:
import pandas as pd
import pickle as pl

# Load all three threshold results
thresholds = [3, 5, 7]
all_results = {}

for days in thresholds:
    with open(f'results_{days}d.pkl', 'rb') as f:
        all_results[days] = pl.load(f)

# Create comparison table
rows = []
for days in thresholds:
    if all_results[days]['avg'] is not None:
        row = {
            'Threshold': f'<{days} days',
            'Accuracy': f"{all_results[days]['avg']['accuracy']:.4f} ± {all_results[days]['std']['accuracy']:.4f}",
            'Precision': f"{all_results[days]['avg']['precision']:.4f} ± {all_results[days]['std']['precision']:.4f}",
            'Recall': f"{all_results[days]['avg']['recall']:.4f} ± {all_results[days]['std']['recall']:.4f}",
            'F1-Score': f"{all_results[days]['avg']['f1']:.4f} ± {all_results[days]['std']['f1']:.4f}",
            'AUC': f"{all_results[days]['avg']['auc']:.4f} ± {all_results[days]['std']['auc']:.4f}"
        }
        rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

# Enhanced visual output with more spacing
print("\n" * 3)
print("=" * 120)
print(" " * 30 + "MORTALITY PREDICTION RESULTS")
print(" " * 25 + "Comparison Across Thresholds")
print("=" * 120)
print("\n")
print(df.to_string(index=False))
print("\n")
print("=" * 120)
print("\n" * 2)

# Save to CSV
df.to_csv('mortality_prediction_results.csv', index=False)
print("✓ Results saved to 'mortality_prediction_results.csv'")
print("\n" * 2)

# Clean table with just mean values
rows_mean = []
for days in thresholds:
    if all_results[days]['avg'] is not None:
        row = {
            'Threshold': f'<{days} days',
            'Accuracy': f"{all_results[days]['avg']['accuracy']:.4f}",
            'Precision': f"{all_results[days]['avg']['precision']:.4f}",
            'Recall': f"{all_results[days]['avg']['recall']:.4f}",
            'F1-Score': f"{all_results[days]['avg']['f1']:.4f}",
            'AUC': f"{all_results[days]['avg']['auc']:.4f}"
        }
        rows_mean.append(row)

df_mean = pd.DataFrame(rows_mean)

print("=" * 100)
print(" " * 35 + "MEAN VALUES ONLY")
print("=" * 100)
print("\n")
print(df_mean.to_string(index=False))
print("\n")
print("=" * 100)
print("\n" * 2)

# Additional summary statistics
print("─" * 100)
print(" " * 35 + "SUMMARY STATISTICS")
print("─" * 100)
for days in thresholds:
    if all_results[days]['avg'] is not None:
        print(f"\n📊 Threshold: <{days} days")
        print(f"   Best Metric: F1-Score = {all_results[days]['avg']['f1']:.4f}")
        print(f"   AUC-ROC: {all_results[days]['avg']['auc']:.4f}")
print("\n" + "─" * 100)
print("\n" * 2)